# 05 · Simulation outputs: from TNG snapshots to gas fields

`skewerjax.simulations` turns the outputs of a hydrodynamical simulation (here a TNG snapshot) into the gas along the
skewers that the optical-depth integrator needs. Layers:

| layer | module | does | arrays |
|---|---|---|---|
| 1 | `readers` | HDF5 I/O and units (`ArepoHDF5Reader`) | NumPy |
| 2 | `gas_cells` | `GasCells`: positions, volumes, physical fields | pytree |
| 3 | `gas_physics` | n_H, T, x_HI, metal ion fractions (CLOUDY) per cell | JAX |
| 4 | `selection` | the cells that can reach the skewers (k-d tree, streamed) | NumPy |
| 5 | `deposition` | which cell fills which piece of every dual segment (Voronoi, or fake_spectra's top hats) | NumPy (default) or JAX (`backend="jax"`) |
| 6 | `absorbers` | pieces → `core.LoSSamples` → `core.optical_depth_from_samples` | JAX |

Why the remaining NumPy steps are NumPy:
- **Reader:** the snapshot is read from HDF5 in streamed blocks (h5py); that is host I/O.
- **Selection:** a periodic k-d tree query (`scipy.spatial.cKDTree(boxsize=...)`) of every streamed block of cells
  against the path points. Its output size, the number of selected cells, depends on the data, which static JAX
  shapes cannot express, and JAX has no periodic tree. It is pure geometry, never differentiated.
- **Deposition:** the default labels sub-points of every dual segment with the same k-d tree and merges the pieces
  with `np.unique`. `voronoi_segments(..., backend="jax")` does the same in JAX with static shapes, with provably exact
  labels and identical pieces. It is slower on a CPU, so it is opt-in.

`SimulationSkewers` runs the six layers for any `ExtractionOptions` / `IntegrationOptions` (defaults: SkewerJAX's).
Data: TNG50-4 snapshot 25 (z = 3.0, 35 cMpc/h, 19.3 M cells), read-only. Runtime: ~1 min. The τ of these skewers
is computed in notebook 06.

**Requires:** skewers (03, with their a(x) from 02) and a simulation snapshot; the atomic table (01) for the ion masses.  
**Independent of:** the toy samplers (04) and any `LineBundle`.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, use_backend, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM,
                                      CMAPS, LABELS, FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()        # dark style: Tol's colour-blind-safe "bright" colours and colour maps, minimum font sizes

In [ ]:
plot_dag(('simulation',)); plt.show()

In [ ]:
from skewerjax.core import LineBundle, SkewerBundle, Cosmology, phys
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, ExtractionOptions, IntegrationOptions, HI, Ion, gas_physics as gp
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
print(f"z = {H.redshift:.4f}, a = {H.a:.6f}, box = {H.box_cMpc:.4f} cMpc, {H.n_gas:,} gas cells in {H.n_files} files, "
      f"a H(z) L = {H.vmax_kms:.1f} km/s")

## 1 · What the integrator needs, and how the snapshot provides it

For line j of species i, along a path γ (review notes, Eq. tau_straight_los; `core.optical_depth.optical_depth_kernel`
L124–L155):

$$\tau_{j}(v_\dagger)=\mathcal C f_j\int_\gamma \frac{n_{i,\rm com}}{a^{2}}\,\phi_{j}(v_\dagger; T, v_\parallel, a)\,dl_{\rm com}
\;\simeq\;\mathcal C f_j\sum_i\sum_{p\in i}\frac{n_{{\rm com},p}\,\ell_p}{a_p^{2}}\,\phi_{j,p}(v_\dagger),\qquad \sum_{p\in i}\ell_p=w_i .$$

So per piece p of gas on the path the integrator needs the comoving density n_com, the temperature T (the width
b = √(2k_BT/m)), the velocity along the path v_∥ (the centre), the scale factor a (the Hubble flow, from the skewer's
a(x)) and the length ℓ_p. The snapshot provides them in six steps:

| step | code | per cell / piece |
|---|---|---|
| 1. read | `readers.ArepoHDF5Reader.to_gas_cells` L213 | x [cMpc] = `Coordinates` × ckpc/h; ρ [g cm⁻³, physical] = `Density` × unit; u [erg/g] = `InternalEnergy` × (km/s)²; v_pec [km/s] = `Velocities` × √a; X_k = `GFM_Metals`; n_e/n_H; SFR; x_HI (stored) |
| 2. cells | `gas_cells.GasCells` | V = M/ρ (comoving), size V^{1/3} (`size_cMpc` L72) |
| 3. physics | `gas_physics.ion_properties` L322 | n_H = ρ X_H/(A_H m_u) (`element_number_density` L124); T = (γ−1) μ m_p u/k_B, μ = 4/(1+3X+4X n_e), X = 0.76 (`temperature` L129); x_HI stored, star-forming cells recomputed (Rahmati+13 at 10⁴ K, `neutral_fraction` L208); metals f_ion = CLOUDY(n_H, T) (`CloudyIonTable`); **n_ion = ρ X_elem/(A m_u) × f_ion** |
| 4. selection | `selection.select_cells_near_paths` | the cells that can touch the paths (periodic k-d tree, streamed) |
| 5. deposition | `deposition.voronoi_segments` L126 | the pieces of every dual segment: cell, exact length ℓ_p, centroid offset (nearest mesh-generating point = Arepo's own cells) |
| 6. absorbers | `absorbers.absorbers_from_components` L163, `absorber_samples` L237 | s_p, n_p, ℓ_p, T_p, v_∥,p = v_pec·t̂; then n_com = n a_snap³, a(s_p) from the frame → `core.LoSSamples` → τ |

The alternative to step 6 is `SimulationSkewers.point_values` (L280): one value per sampling point, the segment
average n_i = Σ_p n_p ℓ_p / w_i with column-weighted T and v. `fields(species)` wraps them as `SampledField`s (04).

## 2 · Per-cell physics (layer 3): the ingredients of the ion densities

Every ion density is n_ion = (ρ X_elem)/(A_elem m_u) × f_ion, with the cell's mass density ρ, the mass fraction
X_elem of the element (`GFM_Metals`), its atomic weight A_elem and the ion fraction f_ion
(`gas_physics.ion_properties`). The ingredients:

- **Hydrogen density.** n_H = ρ X_H/(A_H m_u).
- **Temperature.** T = (γ − 1) μ m_p u/k_B, with the internal energy u, γ = 5/3 and the mean molecular weight
  μ = 4/(1 + 3X + 4X n_e), X = 0.76, n_e = n_e/n_H the stored electron abundance (`gas_physics.temperature`).
- **H I fraction.** TNG stores x_HI from its own ionisation equilibrium with the FG09 UV background, the local AGN
  radiation and the Rahmati et al. (2013) self-shielding factor. SkewerJAX keeps it, except in star-forming cells:
  their stored value follows from the effective temperature of the two-phase medium, so they are recomputed at
  T = 10⁴ K, the cold phase (`neutral_fraction`, model `"stored_sf"`). The recomputation is Rahmati's photoionisation
  equilibrium:

  $$\frac{\Gamma_{\rm phot}}{\Gamma_{\rm UVB}} = 0.98\Big[1+\Big(\frac{n_{\rm H}}{n_{\rm SSh}}\Big)^{1.64}\Big]^{-2.28} + 0.02\Big[1+\frac{n_{\rm H}}{n_{\rm SSh}}\Big]^{-0.84},\qquad
  n_{\rm SSh} = 6.73\times10^{-3}\Big(\frac{\sigma_\nu}{2.49\times10^{-18}}\Big)^{-2/3}\Big(\frac{T}{10^4}\Big)^{0.17}\Big(\frac{\Gamma_{\rm UVB}}{10^{-12}}\Big)^{2/3}\Big(\frac{f_{\rm bar}}{0.17}\Big)^{-1/3}\ {\rm cm^{-3}},$$

  and x_HI solves α_A (1 − x)² n_H = x [Γ_phot + Γ_coll (1 − x) n_H] (case-A recombination α_A(T), collisional
  ionisation Γ_coll(T), n_e = n_HII): x = [B − (B² − 4Aα_A)^{1/2}]/(2A), A = α_A + Γ_coll, B = 2α_A + Γ_phot/n_H + Γ_coll
  (`rahmati_neutral_fraction`). Γ_UVB is the simulation's own H I rate (`TREECOOL_fg_dec11` at the snapshot redshift).
- **Metal ion fractions.** f_ion = CLOUDY(n_H, T): a precomputed table of ionisation equilibrium (FG11 UV background,
  attenuated above 13.6 eV by the same self-shielding factor, 1/10 solar abundances), interpolated linearly in
  redshift and bilinearly in (log n_H, log T) of log f (`CloudyIonTable.fraction`). CLOUDY's density coordinate is
  the hydrogen density, so the cell's n_H enters.

**One temperature for every species.** The same T enters every ion: in the widths, b = (2k_BT/(A m_u))^{1/2}, and in
the ion fractions. The hydrodynamics is single-fluid: ions, neutrals and electrons share one kinetic temperature,
because collisions equilibrate them much faster than the gas evolves. What differs per species is the mass (b_Si/b_H =
(1.008/28.09)^{1/2} = 0.19) and the ionisation model. The exception is star-forming gas: H I is recomputed for its
cold phase (10⁴ K), while its width and the Si III fraction still use the effective temperature (1.5×10⁴–2.4×10⁵ K
here). A per-phase temperature would narrow these lines and change their Si III fraction; it is not done here.

In [ ]:
cells = reader.to_gas_cells(reader.read_rows(0, ["Coordinates", "Masses", "Density", "InternalEnergy", "ElectronAbundance",
                                                    "NeutralHydrogenAbundance", "GFM_Metals", "StarFormationRate", "Velocities"], stop=400_000))
SI3 = Ion("Si", 3)
P_HI, P_Si = gp.ion_properties(cells, HI), gp.ion_properties(cells, SI3)
sf = np.asarray(cells.fields["sfr"]) > 0
table = gp.CloudyIonTable(H.redshift)
from skewerjax.utils.plotting import plot_nH_T_fractions
n_thr = float(np.asarray(P_HI["n_H"])[sf].min())
fig, axs, info = plot_nH_T_fractions(np.asarray(P_HI["n_H"]), np.asarray(P_HI["T"]), np.asarray(P_HI["ion_fraction"]),
                                     np.asarray(P_Si["ion_fraction"]), sf, (table.log_dens, table.log_temp, table.log_fraction_grid(SI3)),
                                     n_thr=n_thr)
add_panel_letters(axs); fig.suptitle("400,000 TNG50-4 cells in the n_H–T plane: H I (stored equilibrium) and Si III (CLOUDY)", fontsize=22)
plt.show()
eos = info["eos"]
print(f"{sf.sum():,} star-forming cells: n_H >= {n_thr:.3f} cm^-3 (fake_spectra's estimate of the threshold: "
      f"{gp.fake_spectra_sf_threshold():.3f}); log T = {eos['slope']:.3f} log n_H + {eos['intercept']:.2f}, scatter {eos['scatter']:.3f} dex")
print(f"median x_HI of star-forming cells: stored {np.median(np.asarray(cells.fields['x_HI_stored'])[sf]):.3f}, "
      f"cold phase (used) {np.median(np.asarray(P_HI['ion_fraction'])[sf]):.3f}; CLOUDY Si III maximum: log f = {info['cloudy_max']['log_f']:.2f} "
      f"at log T = {info['cloudy_max']['log_T']:.2f}")

*Reading the figure.* The contours count cells (1, 10, 100, 1000 per bin), the same in both panels.
- **The sharp cut near n_H ≈ 10⁻¹ cm⁻³** is TNG's star-formation threshold (printed above). Above it the gas is put
  on the Springel & Hernquist effective equation of state: its stored temperature is the pressure of a two-phase
  medium, a single function T(n_H).
- **The thin strip to the right** is that equation of state. The star-forming cells lie on log T ≈ 0.43 log n_H + 4.6
  with a scatter of 0.03 dex (dashed line, fit printed above).
- **Below the threshold**, gas cools down to the floor of photo-heated gas near 10⁴ K: the lower edge of the
  distribution. Shock-heated halo gas fills the upper part (10⁵–10⁷ K), where x_HI ≪ 1 and almost no Si III survives.
- **(b)** Si III is a narrow ridge of the CLOUDY table near 10^4.5 K at these densities. Collisional ionisation removes it
  above, photoionisation at low n_H. So only a small part of the cells carries Si III, mostly dense gas near 10⁴–10⁵ K.

## 3 · Selection → deposition → integration, step by step

Three different grids meet in a simulated spectrum, and the deposition is what connects them:
- **the cells:** one value per Voronoi cell (the region closer to its mesh-generating point than to any other);
- **the sampling points of the `SkewerBundle`:** point i stands for its *dual segment*, between the midpoints with its
  neighbours, of length w_i (the line element of 03);
- **the spectral grid** of v on which τ is evaluated, chosen freely.

The steps are (1) **selection**: the cells close enough to the path to touch it; (2) **deposition**: which cell fills
which piece of every dual segment (its exact length ℓ_p and centroid); (3) **integration**: every piece adds its own
profile, centred where its scale factor a(s) and its velocity v_∥ put it, τ_j(v) = 𝒞 f_j Σ_p n_com,p ℓ_p a_p⁻² φ_j,p(v).
The toy two-dimensional mesh below (a denser filament in the middle) shows all of it at once.

In [ ]:
from skewerjax.utils.plotting import plot_deposition_story
fig, axs, col = plot_deposition_story()
plt.show()
print(f"column along the path: exact {col['exact']:.4f} | cell crossings {col['crossings']:.4f} | segment averages x w_i "
      f"{col['segments']:.4f} ({col['segments'] / col['exact'] - 1:+.1e}) | point values x w_i {col['points']:.4f} "
      f"({col['points'] / col['exact'] - 1:+.1%}) | top-hat discs {col['tophat']:.4f} ({col['tophat'] / col['exact'] - 1:+.1%}) | "
      f"{col['n_pieces']} pieces, {col['n_crossings']} cell crossings")

*Reading the figure.*
- **(a) Selection and deposition.** Each cell is filled with its density. The path is cut into pieces, one per
  (cell, dual segment) pair, and every piece is drawn in the colour of the cell that fills it. The sampling points
  know nothing about the cells: a dual segment usually holds several pieces.
- **(b) How the gas is assigned to the path, and what it does to the column N = ∫n ds.**
  - *Exact pieces* (SkewerJAX's default, `deposition="voronoi"`): every piece keeps its cell's density and its exact
    length, so N is exact for any spacing of the points.
  - *Segment averages* n̄_i = Σ_p n_p ℓ_p / w_i (green bars): the length-weighted mean of the cells that dual segment i
    crosses. Σ_i n̄_i w_i = Σ_p n_p ℓ_p, so they keep the exact column as well. They are what a one-value-per-point
    integrator needs (`SimulationSkewers.point_values`, `fields`: the core path of 04 and 06). What they lose is where
    inside the segment the gas sits, and its velocity structure: all pieces of a segment get one T and one v.
  - *Point values* (red diamonds): whichever cell contains x_i. Their column depends on the spacing.
  - *Top hats* (fake_spectra's kernel, yellow): every cell becomes a sphere (here a disc) of radius V^{1/3} that holds
    its atoms. Spheres overlap and leave gaps, so the density along the path is multi-valued and the column moves.
  - *Cell crossings* (`deposition="voronoi_crossings"`, the bar at the top): the pieces joined across the midpoints,
    one per crossing of a cell. The segmentation follows the cells, not the points, and the column is the same.
- **(c) Integration.** The spectral grid (dotted) is a separate grid. The path enters it only through the cosmology:
  the line centre of a piece is v_c = c ln(a_†/a(s)) + v_∥, approximately H s for a short path. Every piece adds a
  profile there, whatever the pixels are.

**On the snapshot.** Any `SkewerBundle` can be used (`SimulationSkewers.from_bundle`, with its own a(x)); here
three straight skewers along x, sampled every 0.25 km/s of Hubble flow. (1) The selection keeps the cells within reach
of the paths; (2) the deposition finds the exact pieces of the Voronoi cells inside every dual segment; (3) the
absorbers are those pieces, at their centroids. The cell crossings are kept alongside the points
(`SimulationSkewers.crossings()`).

In [ ]:
cosmo = Cosmology(H0=100 * H.h * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m, Omega_r_0=0.0, Omega_m_0=H.omega_m, Omega_k_0=0.0, Omega_lambda_0=H.omega_l)
L = H.box_cMpc
x = np.linspace(0.0, L, int(round(H.vmax_kms / 0.25)) + 1)
points = np.stack([np.stack([x, np.full_like(x, y), np.full_like(x, z)], -1) for y, z in [(10.0, 30.0), (25.0, 5.0), (40.0, 45.0)]])
bundle = SkewerBundle(spatial_points=jnp.asarray(points), a0=H.a, ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
sk = SimulationSkewers.from_bundle(reader, bundle)
print(f"{len(sk.cells):,} cells selected in {sk.selection_s:.1f} s")
seg = sk.segments("voronoi"); A = sk.absorbers(HI)
cr = sk.crossings()
print(f"deposition: {int((seg.cell >= 0).sum()):,} pieces in {seg.cell.shape[0] * seg.cell.shape[1]:,} dual segments "
      f"(up to {seg.cell.shape[-1]} per segment); {int(cr.n_crossings.sum()):,} cell crossings (max gap {cr.diagnostics['max_gap']:.1e} cMpc)")
win = (A.s[0] > 30) & (A.s[0] < 31) & (A.n[0] > 0)
pv = sk.point_values(HI)
fig, axs = plt.subplots(2, 1, figsize=(14, 7), sharex=True, layout="constrained")
axs[0].semilogy(sk.s[0], pv["n"][0], color=FG, lw=1, label=r"segment average $\sum_p n_p\ell_p / w_i$")
axs[0].semilogy(A.s[0][win], A.n[0][win], ".", color=EC["density"], ms=6, label="pieces, at their centroids")
live = cr.cell[0] >= 0
for lo_, hi_, c_ in zip(cr.s_lo[0][live], cr.s_hi[0][live], cr.cell[0][live]):
    if hi_ > 30 and lo_ < 31:
        axs[0].plot([lo_, hi_], [sk.properties(HI)["n_ion"][c_]] * 2, color=EC["crossings"], lw=3, alpha=0.8)
axs[0].plot([], [], color=EC["crossings"], lw=3, label="cell crossings")
axs[1].semilogy(A.s[0][win], A.T[0][win], ".", color=EC["temperature"], ms=4)
axs[0].set(xlim=(30, 31), ylabel=r"$n_{\rm HI}$ [cm$^{-3}$]"); axs[1].set(xlabel="path length s [cMpc]", ylabel="T [K]"); axs[0].legend()
add_panel_letters(axs); fig.suptitle("Voronoi deposition: piecewise-constant gas, one value per cell crossing", fontsize=22); plt.show()

## 4 · The column does not depend on the sampling

The same three skewers sampled 0.25, 2, 8 and 32 km/s apart (in Hubble flow), with the cells already selected. The
Voronoi deposition integrates every dual segment exactly, so the total N_HI of each skewer is the same at every
spacing. The point deposition (`deposition="points"`: one value per point, the top-hat spheres containing it, no
line element; overlapping spheres are summed) samples the gas instead: its column is off even at 0.25 km/s and drifts
as the points thin out. The cell crossings (`deposition="voronoi_crossings"`) give the Voronoi column too.

In [ ]:
rows = []
for dv_pt in (0.25, 2.0, 8.0, 32.0):
    xs = np.linspace(0.0, L, int(round(H.vmax_kms / dv_pt)) + 1)
    pts = np.stack([np.stack([xs, np.full_like(xs, p[0, 1]), np.full_like(xs, p[0, 2])], -1) for p in points])
    skc = SimulationSkewers(reader, pts, cells=sk.cells)
    N = {kind: skc.column_per_pixel(HI, ExtractionOptions(deposition=kind), nbins=512).sum(-1) for kind in ("voronoi", "points", "voronoi_crossings")}
    rows.append((dv_pt, xs.size, N["voronoi"], N["points"], N["voronoi_crossings"]))
N0 = rows[0][2]
fig, ax = plt.subplots(figsize=FIGSIZES["single"], layout="constrained")
for k in range(3):
    c = list(TOL.values())[k]
    ax.plot([r[0] for r in rows], [r[2][k] / N0[k] for r in rows], "o-", color=c, label=f"skewer {k}, Voronoi segments")
    ax.plot([r[0] for r in rows], [r[3][k] / N0[k] for r in rows], "s--", color=c, label=f"skewer {k}, point values")
ax.set(xscale="log", yscale="log", xlabel="spacing of the sampling points [km/s of Hubble flow]",
       ylabel=r"$N_{\rm HI}$ / Voronoi $N_{\rm HI}$ at 0.25 km/s")
ax.axhline(1, color=FG, lw=1); ax.legend(fontsize=15, ncol=2)
ax.set_title("Total H I column per skewer: exact for Voronoi segments at any spacing, not for point values", fontsize=18); plt.show()
print(f"Voronoi: max |N / N(0.25 km/s) - 1| over spacings and skewers = {max(np.abs(r[2] / N0 - 1).max() for r in rows):.1e}; "
      f"cell crossings vs Voronoi pieces: {max(np.abs(r[4] / r[2] - 1).max() for r in rows):.1e}")
for dv_pt, n, Nv, Np, Nc in rows:
    print(f"spacing {dv_pt:5.2f} km/s ({n:6d} points): log10 N_HI Voronoi {np.round(np.log10(Nv), 6)} | points {np.round(np.log10(Np), 4)}")

**Options.** `ExtractionOptions`: `deposition` ("voronoi" | "tophat" | "points" | "voronoi_crossings"), `mass_unit` ("amu" | "proton"),
`T_constants`, `b_constants` ("codata" | "fake_spectra"), `x_HI_model` ("stored_sf" | "stored" | "fake_sf" |
"rahmati_all"), `cloudy_density` ("hydrogen" | "fake_spectra"), `cloudy_interpolation` ("linear" | "cubic").
`IntegrationOptions`: `frame` ("exact" | "snapshot"), `normalisation`, `detuning` ("exact" | "linear"),
`periodic`. The non-default values reproduce fake_spectra (notebook 08). Checks:
`scripts/validation/check_tng_fields.py`, `check_tng_optical_depth.py`.